<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1


In [4]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
march_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = f"{base}/fact_content_daily_performance/month=2026-04/*.parquet"

march_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_sum_position) AS march_sum_position
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
march_df["avg_position"] = march_df["march_sum_position"] / march_df["march_impressions"]
march_df["ctr"] = march_df["march_clicks"] / march_df["march_impressions"].replace(0, pd.NA)

april_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

merged = march_df.merge(april_df, on=["content_hash_id", "client_hash_id"], how="inner")
merged["is_declining"] = (merged["april_clicks"] < merged["march_clicks"]).astype(int)
print(f"Shape: {merged.shape}, base rate: {merged['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (158549, 9), base rate: 0.279


## 1. Question

*The research question and the decision it supports.*

In [1]:
"""
Research question: Which content pages in FlyRank's Lane 2
(Refresh/Content Opportunity Scoring) are most likely to see a decline
in organic clicks next month, so a content reviewer can prioritize a
limited review budget?

Decision this supports: a content strategist/SEO reviewer has capacity
to manually review a limited number of pages each cycle (e.g. the top
50). This work ranks pages so that limited human attention goes to the
pages most likely to need it, rather than being spent at random or on
whichever pages happen to be top-of-mind.

Cost of a wrong call: a false positive costs a reviewer's time on a
page that didn't need it; a false negative lets a genuinely declining
page go unnoticed, which compounds silently over time -- the costlier
error, established back in the original lane-framing work.
"""

"\nResearch question: Which content pages in FlyRank's Lane 2 \n(Refresh/Content Opportunity Scoring) are most likely to see a decline \nin organic clicks next month, so a content reviewer can prioritize a \nlimited review budget?\n\nDecision this supports: a content strategist/SEO reviewer has capacity \nto manually review a limited number of pages each cycle (e.g. the top \n50). This work ranks pages so that limited human attention goes to the \npages most likely to need it, rather than being spent at random or on \nwhichever pages happen to be top-of-mind.\n\nCost of a wrong call: a false positive costs a reviewer's time on a \npage that didn't need it; a false negative lets a genuinely declining \npage go unnoticed, which compounds silently over time -- the costlier \nerror, established back in the original lane-framing work.\n"

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
"""
Data release: FlyRank/internship-warehouse (Hugging Face, gated, ~79M
rows across all months). This work used two month partitions from
fact_content_daily_performance: month=2026-03 (features) and
month=2026-04 (label construction only).

Tables used: fact_content_daily_performance (report_date x client x
content grain). dim_content and dim_clients were consulted during
earlier data-contract work but are not model inputs in the final
pipeline.

Date windows: March 2026 for all features (clicks, impressions,
position, CTR). April 2026 used ONLY to determine whether clicks
declined -- never merged into the feature set.

Filtering applied: rows without gsc_data_available = TRUE were
excluded, since many rows in the raw fact table are zero-filled
placeholders for dates before a client's tracking began, not genuine
zero-activity days. After filtering, ~55 of 104 total clients had
usable March data -- roughly half the client base is not represented
in this analysis, a real coverage limitation carried through to the
Limitations section.

What was excluded and why (public-safe): no client names, domains,
URLs, page titles, or raw search queries appear anywhere in this work
-- only pseudonymized content_hash_id / client_hash_id values, used
strictly for grouping and joining, never as model features. FlyRank's
own product-decision fields (e.g. health-score-style flags,
optimization-eligible dates) were deliberately excluded as features,
since using them would mean the model learns to reproduce an existing
product decision rather than finding independent signal.
"""

"\nData release: FlyRank/internship-warehouse (Hugging Face, gated, ~79M \nrows across all months). This work used two month partitions from \nfact_content_daily_performance: month=2026-03 (features) and \nmonth=2026-04 (label construction only).\n\nTables used: fact_content_daily_performance (report_date x client x \ncontent grain). dim_content and dim_clients were consulted during \nearlier data-contract work but are not model inputs in the final \npipeline.\n\nDate windows: March 2026 for all features (clicks, impressions, \nposition, CTR). April 2026 used ONLY to determine whether clicks \ndeclined -- never merged into the feature set.\n\nFiltering applied: rows without gsc_data_available = TRUE were \nexcluded, since many rows in the raw fact table are zero-filled \nplaceholders for dates before a client's tracking began, not genuine \nzero-activity days. After filtering, ~55 of 104 total clients had \nusable March data -- roughly half the client base is not represented \nin this 

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["march_clicks", "march_impressions", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

# Honest, client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Confirm no client overlap
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Client overlap check: {len(train_clients & test_clients)} (should be 0)")

# Train both models
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

print("Models trained on the honest, client-grouped split.")

Client overlap check: 0 (should be 0)
Models trained on the honest, client-grouped split.


In [6]:
"""
Assumptions: content performance in one month carries some signal about
the next month's trajectory; a page's own trailing behavior (clicks,
impressions, position, CTR) is a reasonable, observable starting point
for a first model, even without external factors like seasonality or
competitor changes.

Features (4, all knowable at the March decision point): march_clicks,
march_impressions, avg_position (derived as SUM(position)/SUM(impressions),
corrected after an initial averaging bug was caught during data-contract
work), and ctr (march_clicks/march_impressions).

Label definition: is_declining = 1 if April clicks < March clicks, for
the same content+client pair, else 0. Built from genuine FORWARD data
(April), never from the same period as the features -- avoiding the
exact same-period leakage trap identified during an earlier data-contract
audit, where a label-derived column was deliberately added as a "feature"
and produced a dishonest, near-perfect score.

Baseline: a transparent, hand-written rule from an earlier stage of this
work -- flagging pages with march_impressions >= 100 AND avg_position >=
20, scored by impressions. No fitted weights, fully human-readable.

Validation design: GroupShuffleSplit grouped by client_hash_id, 30% test
size, confirmed 0 client overlap between train and test. This was
deliberately chosen over a naive random row split after a controlled
comparison showed the naive split reports a materially higher,
untrustworthy score (see Results).

Leakage checks performed: (1) temporal check confirming no April data
appears in any feature; (2) same-period trap check confirming no feature
is mathematically derived from the label itself within the same time
window; (3) feature-importance sanity check confirming no single feature
dominates suspiciously (all four features contribute, none above ~0.5
importance); (4) confirmation that no FlyRank product-decision fields
were used as inputs.
"""

'\nAssumptions: content performance in one month carries some signal about \nthe next month\'s trajectory; a page\'s own trailing behavior (clicks, \nimpressions, position, CTR) is a reasonable, observable starting point \nfor a first model, even without external factors like seasonality or \ncompetitor changes.\n\nFeatures (4, all knowable at the March decision point): march_clicks, \nmarch_impressions, avg_position (derived as SUM(position)/SUM(impressions), \ncorrected after an initial averaging bug was caught during data-contract \nwork), and ctr (march_clicks/march_impressions).\n\nLabel definition: is_declining = 1 if April clicks < March clicks, for \nthe same content+client pair, else 0. Built from genuine FORWARD data \n(April), never from the same period as the features -- avoiding the \nexact same-period leakage trap identified during an earlier data-contract \naudit, where a label-derived column was deliberately added as a "feature" \nand produced a dishonest, near-perfect 

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [7]:
# --- Baseline rule, applied to the test set ---
test_df = merged.iloc[test_idx].copy()
has_volume = (test_df["march_impressions"] >= 100).astype(int)
weak_position = (test_df["avg_position"] >= 20).astype(int)
baseline_score = has_volume * weak_position * test_df["march_impressions"]

# --- Model scores on the honest test set ---
logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]

base_rate = y_test.mean()
honest_results = {
    "Base rate (random)":       base_rate,
    "Baseline rule":            precision_at_k(baseline_score, y_test.values, 50),
    "Logistic Regression":      precision_at_k(logreg_scores, y_test.values, 50),
    "Random Forest":            precision_at_k(rf_scores, y_test.values, 50),
}

# --- Naive random split, for the honest-vs-naive contrast ---
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X, y, test_size=0.3, random_state=42
)
rf_bad = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_bad.fit(X_train_bad, y_train_bad)
bad_scores = rf_bad.predict_proba(X_test_bad)[:, 1]
naive_p50 = precision_at_k(bad_scores, y_test_bad.values, 50)

print("=== Honest, client-grouped split (trusted result) ===")
for name, score in honest_results.items():
    print(f"{name:25s} {score:.3f}")

print(f"\n=== Naive random split (for comparison only) ===")
print(f"{'Random Forest (naive split)':25s} {naive_p50:.3f}")

=== Honest, client-grouped split (trusted result) ===
Base rate (random)        0.331
Baseline rule             0.640
Logistic Regression       0.660
Random Forest             0.800

=== Naive random split (for comparison only) ===
Random Forest (naive split) 0.900


In [9]:
"""
Results (same test set, same metric, same notebook run, per the
training-honest-models skill's non-negotiable rule):

Base rate (random guessing):    0.331
Baseline rule (hand-written):   0.640
Logistic Regression:            0.660
Random Forest:                  0.800

Random Forest OBSERVES the strongest ranking performance at Precision@50
on this split, beating both the baseline rule and Logistic Regression.
This is a single-split result, not a cross-validated average, and is
treated as DECISION-SUPPORT for which model to develop further -- not
proof this exact margin holds on every future split or time period.

Honest-vs-naive contrast: the same Random Forest model, evaluated
instead on a naive random row-level split, reports Precision@50 = 0.900
-- a full 10 points higher, with clients overlapping between train and
test. This is not a stronger model; it is a data leak. The 0.800
grouped-split result is the number reported and trusted throughout this
paper.

Note on reproducibility: earlier notebooks in this project (validation
audit, action playbook) recorded 0.780/0.940 for this same comparison,
run with the same random seed. This run instead shows 0.800/0.900 -- a
small, expected shift, consistent with this repo's own documented note
that tree-ensemble results can move a few points between runs/library
versions. The consistent finding across every run is the same: Random
Forest clearly beats the baseline rule, and the naive split materially
overstates performance versus the honest, grouped split -- the specific
decimal is secondary to that pattern.
"""

"\nResults (same test set, same metric, same notebook run, per the \ntraining-honest-models skill's non-negotiable rule):\n\nBase rate (random guessing):    0.331\nBaseline rule (hand-written):   0.640\nLogistic Regression:            0.660\nRandom Forest:                  0.800\n\nRandom Forest OBSERVES the strongest ranking performance at Precision@50 \non this split, beating both the baseline rule and Logistic Regression. \nThis is a single-split result, not a cross-validated average, and is \ntreated as DECISION-SUPPORT for which model to develop further -- not \nproof this exact margin holds on every future split or time period.\n\nHonest-vs-naive contrast: the same Random Forest model, evaluated \ninstead on a naive random row-level split, reports Precision@50 = 0.900 \n-- a full 10 points higher, with clients overlapping between train and \ntest. This is not a stronger model; it is a data leak. The 0.800 \ngrouped-split result is the number reported and trusted throughout this \

## 5. Limitations

*What this work cannot claim.*

In [10]:
"""
What this work cannot claim:

1. Not causal. This is observational, cross-sectional analysis across
one warehouse snapshot -- it shows an ASSOCIATION between a page's
March signals and its April click trajectory, not proof that any
specific action (e.g. a content refresh) will cause a specific outcome.
No controlled experiment was run.

2. Limited time window. Trained and validated on one March->April 2026
transition only. Not tested on other months, seasons, or years --
results may not generalize to different time periods without
re-validation.

3. Partial client coverage. Only ~55 of 104 total clients had usable
March data after filtering for genuine tracking availability -- roughly
half the client base is not represented, and results may not generalize
evenly across all clients.

4. Single-split result. The headline Precision@50 numbers come from one
train/test split, not a cross-validated average across multiple splits
-- per this repo's own documented note, tree-ensemble numbers can shift
a few points run to run.

5. Feature scope. Based on 4 observable signals only. Does not account
for real-world causes of decline like site migrations, manual
penalties, seasonal demand shifts, or competitor changes -- a human
reviewer supplies that context; the model cannot.

6. Errors concentrate on already near-zero-click content, where the
model's top feature (march_clicks) carries little signal because
there's little room left to "decline" in absolute terms -- a known,
named weak spot, not a hidden one.
"""

'\nWhat this work cannot claim:\n\n1. Not causal. This is observational, cross-sectional analysis across \none warehouse snapshot -- it shows an ASSOCIATION between a page\'s \nMarch signals and its April click trajectory, not proof that any \nspecific action (e.g. a content refresh) will cause a specific outcome. \nNo controlled experiment was run.\n\n2. Limited time window. Trained and validated on one March->April 2026 \ntransition only. Not tested on other months, seasons, or years -- \nresults may not generalize to different time periods without \nre-validation.\n\n3. Partial client coverage. Only ~55 of 104 total clients had usable \nMarch data after filtering for genuine tracking availability -- roughly \nhalf the client base is not represented, and results may not generalize \nevenly across all clients.\n\n4. Single-split result. The headline Precision@50 numbers come from one \ntrain/test split, not a cross-validated average across multiple splits \n-- per this repo\'s own doc

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [11]:
# Re-generate the ranked queue with the final model from this notebook
merged["decline_risk_score"] = rf.predict_proba(X)[:, 1]

merged["reason_code"] = "monitor"
merged.loc[(merged["decline_risk_score"] >= 0.6) & (merged["ctr"] < merged["ctr"].median()), "reason_code"] = "high_risk_low_ctr"
merged.loc[(merged["decline_risk_score"] >= 0.6) & (merged["avg_position"] >= 20), "reason_code"] = "high_risk_weak_position"
merged.loc[(merged["decline_risk_score"] >= 0.4) & (merged["decline_risk_score"] < 0.6), "reason_code"] = "moderate_risk_watch"

action_map = {
    "high_risk_low_ctr": "review_for_ctr_fix",
    "high_risk_weak_position": "review_for_position_and_ctr",
    "moderate_risk_watch": "monitor_next_cycle",
    "monitor": "no_action_needed",
}
merged["action"] = merged["reason_code"].map(action_map)

ranked_queue = merged.sort_values("decline_risk_score", ascending=False).reset_index(drop=True)
print(ranked_queue["reason_code"].value_counts())

reason_code
monitor                    142159
moderate_risk_watch          8787
high_risk_weak_position      7603
Name: count, dtype: int64


In [12]:
"""
Ranked recommendations (full detail in the Content Action Playbook,
work/notebooks/w07_action_playbook.ipynb):

Pages are ranked by decline_risk_score and assigned one of four reason
codes: high_risk_weak_position (review_for_position_and_ctr),
high_risk_low_ctr (review_for_ctr_fix), moderate_risk_watch
(monitor_next_cycle), and monitor (no_action_needed).

Intended use: decision-support for a reviewer triaging a limited weekly
review budget -- not a fully-automated action system. Every flagged
page requires human review before any change is made.

No-go list -- must NEVER be automated: auto-publishing content changes,
auto-messaging clients about a flagged page, using this score as an
input to billing or contract decisions, or treating "monitor" as proof
a page has been reviewed and cleared.

Monitoring triggers: re-validate after 3 months, or sooner if the base
rate of decline shifts materially, if a spot-check of the top-50 queue
falls meaningfully below the validated 0.800 Precision@50, or if client
data coverage drops further from the current ~55 of 104.
"""

'\nRanked recommendations (full detail in the Content Action Playbook, \nwork/notebooks/w07_action_playbook.ipynb):\n\nPages are ranked by decline_risk_score and assigned one of four reason \ncodes: high_risk_weak_position (review_for_position_and_ctr), \nhigh_risk_low_ctr (review_for_ctr_fix), moderate_risk_watch \n(monitor_next_cycle), and monitor (no_action_needed).\n\nIntended use: decision-support for a reviewer triaging a limited weekly \nreview budget -- not a fully-automated action system. Every flagged \npage requires human review before any change is made.\n\nNo-go list -- must NEVER be automated: auto-publishing content changes, \nauto-messaging clients about a flagged page, using this score as an \ninput to billing or contract decisions, or treating "monitor" as proof \na page has been reviewed and cleared.\n\nMonitoring triggers: re-validate after 3 months, or sooner if the base \nrate of decline shifts materially, if a spot-check of the top-50 queue \nfalls meaningfully b

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.